In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision-figures"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)


import pandas as pd
import scanpy as sc

from spatial_tcr.clonal_expansion import find_avbv_clones
from spatial_tcr.plotting import (
    plot_side_by_side_counts,
)
from spatial_tcr.tcr import get_tcr_genes

## Perform analysis on unmerged TRVs


In [ ]:
path = "data/xenium/processed/05.2-kidney_tcr_infilrate.h5ad"
adata = sc.read_h5ad(path)
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)
find_avbv_clones(adata, av_genes, bv_genes, layer="counts")

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()

In [ ]:
df_1 = ad_t.obs.loc[ad_t.obs["avbv_clone"].isna(), "av_clone"].value_counts().to_frame()
df_2 = (
    ad_t.obs.loc[~ad_t.obs["avbv_clone"].isna(), "av_clone"].value_counts().to_frame()
)
print(df_1.shape, df_2.shape)
av_counts = pd.concat([df_1, df_2], axis=1).fillna(0)
av_counts.columns = ["single", "paired"]

plot_side_by_side_counts(
    av_counts,
    normalize=True,
    title="TRAV Gene Expression Distribution in Single vs Paired Cells",
    annotate_per_category=False,
    figsize=(7, 4),
)

In [ ]:
df_1 = ad_t.obs.loc[ad_t.obs["avbv_clone"].isna(), "bv_clone"].value_counts().to_frame()
df_2 = (
    ad_t.obs.loc[~ad_t.obs["avbv_clone"].isna(), "bv_clone"].value_counts().to_frame()
)
print(df_1.shape, df_2.shape)
bv_counts = pd.concat([df_1, df_2], axis=1).fillna(0)
bv_counts.columns = ["single", "paired"]

plot_side_by_side_counts(
    bv_counts,
    normalize=True,
    title="TRBV Gene Expression Distribution in Single vs Paired Cells",
    annotate_per_category=False,
    figsize=(7, 4),
)